In [ ]:
import os 
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [ ]:
hf_token = os.getenv("HF_TOKEN")

In [ ]:
hf_token

In [ ]:
loader = TextLoader("hello.txt")
doc = loader.load()

print(type(doc))

In [ ]:
print(type(doc))
print(type(doc[0]))
print(doc[0])

In [ ]:
text_splitter = CharacterTextSplitter(chunk_size = 500, chunk_overlap = 30)

In [ ]:
documents = text_splitter.split_documents(doc)

In [ ]:
print(len(documents))

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [ ]:
from langchain_ollama import ChatOllama

In [ ]:
llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

In [ ]:
response = llm.invoke("Who are you?")

print(response.content)

In [ ]:
db = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

In [ ]:
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)

In [ ]:
query = "What is my name?"

docs = retriever.invoke(query)

docs

In [ ]:
for doc in docs:
    print(doc.page_content)

In [ ]:
context = "\n".join(
    [doc.page_content for doc in docs]
)

print(context)

In [ ]:
query = "What is my name?"

prompt = f"""
You are an AI Assistant.

Answer ONLY using the context below.

If the answer is not present, reply:

I don't know.

Context:
{context}

Question:
{query}

Answer:
"""

In [ ]:
response = llm.invoke(prompt)

print(response.content)

In [ ]:
while True:

    query = input("Ask your question: ")

    if query.lower() == "exit":
        break

    docs = retriever.invoke(query)

    context = "\n".join(
        [doc.page_content for doc in docs]
    )

    prompt = f"""
You are an AI Assistant.

Answer ONLY using the context below.

If the answer isn't available, say "I don't know."

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)

    print("\nAnswer:")
    print(response.content)
    print("=" * 60)